# Dự báo Doanh số Thương mại Điện tử
## Giai đoạn 2 — Linear Regression (Baseline Model)
---

## Import thư viện

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## Nhập lại Giai đoạn 1 (Metrics + Scaler)
> Copy từ `giai_doan_1.ipynb` hoặc chạy lại cell bên dưới.

In [ ]:
# ── Evaluation Metrics ────────────────────────────────────────
def calculate_mse(y_true, y_pred):
    n = len(y_true)
    return (1 / n) * np.sum((y_true - y_pred) ** 2)

def calculate_rmse(y_true, y_pred):
    return np.sqrt(calculate_mse(y_true, y_pred))

def calculate_mae(y_true, y_pred):
    n = len(y_true)
    return (1 / n) * np.sum(np.abs(y_true - y_pred))

def calculate_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

# ── CustomStandardScaler ──────────────────────────────────────
class CustomStandardScaler:
    def __init__(self):
        self.mean_ = None
        self.std_  = None

    def fit(self, X):
        self.mean_ = np.mean(X, axis=0)
        self.std_  = np.std(X, axis=0)
        self.std_[self.std_ == 0] = 1
        return self

    def transform(self, X):
        if self.mean_ is None:
            raise RuntimeError("Cần gọi fit() trước khi transform()")
        return (X - self.mean_) / self.std_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

print("Đã load xong Giai đoạn 1 (Metrics + Scaler)")

---
## Giai đoạn 2 — LinearRegression
Tìm siêu mặt phẳng tối ưu `ŷ = X·W + b` thông qua thuật toán **Gradient Descent**.

In [ ]:
class LinearRegression:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr           = learning_rate
        self.n_iterations = n_iterations
        self.W            = None  # trọng số (weights)
        self.b            = None  # hệ số chặn (bias)
        self.loss_history = []    # theo dõi quá trình hội tụ

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Khởi tạo W và b bằng 0
        self.W = np.zeros(n_features)
        self.b = 0.0
        self.loss_history = []

        for i in range(self.n_iterations):
            # Forward pass: tính y_hat
            y_pred = X @ self.W + self.b

            # Tính gradients từ MSE
            error = y_pred - y
            dW = (2 / n_samples) * (X.T @ error)
            db = (2 / n_samples) * np.sum(error)

            # Cập nhật trọng số
            self.W -= self.lr * dW
            self.b -= self.lr * db

            # Lưu loss mỗi 100 epochs
            if i % 100 == 0:
                loss = np.mean(error ** 2)
                self.loss_history.append(loss)

        return self

    def predict(self, X):
        if self.W is None:
            raise RuntimeError("Cần gọi fit() trước khi predict()")
        return X @ self.W + self.b

print("Đã định nghĩa xong LinearRegression")

---
## Pipeline: Load → Chuẩn hóa → Huấn luyện → Đánh giá

In [ ]:
# ── LOAD DỮ LIỆU ──────────────────────────────────────────────
# Thay đường dẫn cho đúng với máy của bạn
X_train = pd.read_csv("../dataset_ready/X_train.csv").values.astype(float)
X_test  = pd.read_csv("../dataset_ready/X_test.csv").values.astype(float)
y_train = pd.read_csv("../dataset_ready/y_train.csv").values.ravel().astype(float)
y_test  = pd.read_csv("../dataset_ready/y_test.csv").values.ravel().astype(float)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

assert not np.isnan(X_train).any(), "X_train có giá trị NaN!"
assert not np.isnan(y_train).any(), "y_train có giá trị NaN!"
print("\nDữ liệu hợp lệ, không có NaN")

In [ ]:
# ── CHUẨN HÓA ─────────────────────────────────────────────────
scaler     = CustomStandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Chuẩn hóa hoàn tất")

In [ ]:
# ── HUẤN LUYỆN ────────────────────────────────────────────────
model = LinearRegression(learning_rate=0.01, n_iterations=1000)
model.fit(X_train_sc, y_train)

print("── Trọng số sau huấn luyện ──")
print(f"W : {model.W}")
print(f"b : {model.b:.4f}")

In [ ]:
# ── ĐÁNH GIÁ ──────────────────────────────────────────────────
y_pred = model.predict(X_test_sc)

print("── Kết quả Linear Regression trên tập Test ──")
print(f"MSE  : {calculate_mse(y_test, y_pred):.2f}")
print(f"RMSE : {calculate_rmse(y_test, y_pred):.2f}")
print(f"MAE  : {calculate_mae(y_test, y_pred):.2f}")
print(f"R²   : {calculate_r2(y_test, y_pred):.4f}")

In [ ]:
# ── BIỂU ĐỒ 1: Loss Curve ─────────────────────────────────────
plt.figure(figsize=(8, 4))
epochs = [i * 100 for i in range(len(model.loss_history))]
plt.plot(epochs, model.loss_history, color='steelblue', linewidth=2)
plt.xlabel("Epochs")
plt.ylabel("MSE Loss")
plt.title("Loss Curve — Quá trình hội tụ của Gradient Descent")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── BIỂU ĐỒ 2: Actual vs Predicted ───────────────────────────
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white', linewidth=0.5)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Dự báo hoàn hảo')
plt.xlabel("Doanh số thực tế (y_test)")
plt.ylabel("Doanh số dự báo (y_pred)")
plt.title("Actual vs Predicted — Linear Regression")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()